# ISIC 2019 Bronze tables and folders

This notebook creates the ISIC 2019 dataset-specific Bronze table and volume folder layout.

Run this after `notebooks/00_setup_storage.ipynb` and before `10_bronze_ingest.ipynb`. It also prints the exact landing paths where the source archives must exist.


In [ ]:
%run ../_setup_env


In [ ]:
from pathlib import Path

from data_platform.setup import build_layout, load_dataset_config, load_yaml_config

DATASET_CONFIG = load_dataset_config(CONFIG_ROOT / "datasets" / "isic_2019.yaml")
STORAGE_CONFIG = load_yaml_config(CONFIG_ROOT / "storage.yaml")
DATASET_CONFIG["landing_root"] = STORAGE_CONFIG["landing_root"]
DATASET_CONFIG["storage_root"] = STORAGE_CONFIG["storage_root"]
layout = build_layout(DATASET_CONFIG)
landing_paths = layout["landing_paths"]
bronze_paths = layout["bronze_paths"]
bronze_tables = layout["bronze_tables"]

ARCHIVES = [
    {
        "source_split": "train",
        "archive_filename": "ISIC-2019-train-images.zip",
        "metadata_filename": "metadata.csv", # expected metadata filename inside archive
    },
    {
        "source_split": "test",
        "archive_filename": "ISIC-2019-test-images.zip",
        "metadata_filename": "metadata.csv", # expected metadata filename inside archive
    },
]

for archive in ARCHIVES:
    source_split = archive["source_split"]
    archive["archive_dbfs_path"] = f"{landing_paths['dataset_archive'].rstrip('/')}/{archive['archive_filename']}"
    archive["archive_local_path"] = Path("/dbfs") / archive["archive_dbfs_path"].removeprefix("dbfs:/")
    archive["bronze_images_path"] = f"{bronze_paths['images'].rstrip('/')}/{source_split}"
    archive["metadata_dir_path"] = f"{bronze_paths['metadata'].rstrip('/')}/{source_split}"
    archive["metadata_target_path"] = f"{archive['metadata_dir_path'].rstrip('/')}/{archive['metadata_filename']}"

print(layout)
for archive in ARCHIVES:
    print(f"{archive['source_split']} archive upload path: {archive['archive_dbfs_path']}")
    print(f"{archive['source_split']} Bronze images path: {archive['bronze_images_path']}")
    print(f"{archive['source_split']} Bronze metadata path: {archive['metadata_target_path']}")


## Create Dataset Objects

Create the ISIC 2019 Bronze source table and dataset-specific landing, image, and metadata folders.


In [ ]:
spark.sql(
    """
    CREATE TABLE IF NOT EXISTS bronze.isic_2019_source_metadata (
      image_id STRING,
      source_split STRING,
      source_uri STRING,
      attribution STRING,
      copyright_license STRING,
      age_approx STRING,
      anatom_site_1 STRING,
      anatom_site_2 STRING,
      anatom_site_3 STRING,
      anatom_site_4 STRING,
      anatom_site_5 STRING,
      anatom_site_special STRING,
      clin_size_long_diam_mm STRING,
      concomitant_biopsy STRING,
      dermoscopic_type STRING,
      diagnosis_1 STRING,
      diagnosis_2 STRING,
      diagnosis_3 STRING,
      diagnosis_4 STRING,
      diagnosis_5 STRING,
      diagnosis_confirm_type STRING,
      family_hx_mm STRING,
      image_type STRING,
      lesion_id STRING,
      melanocytic STRING,
      patient_id STRING,
      personal_hx_mm STRING,
      sex STRING,
      source_checksum STRING,
      ingestion_run_id STRING,
      ingested_at TIMESTAMP
    )
    USING DELTA
    """
)
dbutils.fs.mkdirs(landing_paths["dataset_archive"])
for path in bronze_paths.values():
    dbutils.fs.mkdirs(path)
for archive in ARCHIVES:
    dbutils.fs.mkdirs(archive["bronze_images_path"])
    dbutils.fs.mkdirs(archive["metadata_dir_path"])

display(spark.sql("SHOW TABLES IN bronze"))
display(dbutils.fs.ls(landing_paths["archives"]))
display(dbutils.fs.ls(bronze_paths["root"]))


## Upload Checklist

Upload these archives to the paths printed above before running `10_bronze_ingest.ipynb`:

- `ISIC-2019-train-images.zip`
- `ISIC-2019-test-images.zip`
